In [ ]:
!pip install dotenv openai pandas openpyxl requests chromadb

In [9]:
from agno.agent import Agent
from agno.models.message import Message
from agno.db.sqlite import SqliteDb
from agno.tools.serper import SerperTools
from agno.models.azure import AzureOpenAI
from agno.knowledge import Knowledge
from agno.vectordb.chroma import ChromaDb
import os, json
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  

In [2]:
load_dotenv()
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [11]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


### Load Taxonomy as CSV and convert to JSON

In [7]:
df_taxonomy = pd.read_excel(TAXONOMY_FILE, sheet_name="custom_genre")
taxonomy_json = df_taxonomy.to_dict(orient='records')
taxonomy_json

[{'cluster_id': 5,
  'custom_genre': 'Clever Heist Capers',
  'description': 'This genre unites films centered around smart, stylish criminals and their intricate schemes, blending suspense, wit, and high-stakes cons or heists. These movies focus on the thrill of the plan, the dynamics within the crew, and the clever twists that keep audiences guessing.',
  'exemplar_overview': 'When a seasoned thief assembles a diverse team to pull off the ultimate heist, they must outsmart rival criminals and law enforcement alike. As tensions rise and loyalties are tested, the crew navigates double-crosses and unexpected obstacles to claim their prize and secure their freedom.',
  'included_movies': '[\'Heist\', \'To Catch a Thief\', \'Trading Places\', \'Heat\', "White Men Can\'t Jump", \'Gone in Sixty Seconds\', \'The Italian Job\', \'The Town\', \'Small Time Crooks\', "Ocean\'s Eleven", \'Inside Man\', \'Logan Lucky\']'},
 {'cluster_id': 8,
  'custom_genre': 'Global Espionage Thrillers',
  'descr

## Load PDF with PAGE-LEVEL Chunking (Sandbox Method)

**Key Difference**: Instead of splitting by characters, we create 2-page chunks with 1-page overlap.

In [8]:
knowledge = Knowledge(
    vector_db=ChromaDb(collection="movie_news", path="tmp/chromadb"),
)

# Load content
await knowledge.ainsert(path=OSCARS_PDF)

INFO Adding content from path, eb822c93-d42f-522a-87ef-92dd90d4207a, None,                                         
     ../data/input/The_98th_Academy_Awards_2026.pdf, None

INFO Async Upserting 11 documents

In [12]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
agent_instructions = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
AGENT_SYSTEM_PROMPT =  agent_instructions + "\n\nCustom Genre\n" + json.dumps(taxonomy_json, indent=2)
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)




System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to make a pers

In [13]:
agent = Agent(
    model=model,
    db=agent_db,
    knowledge=knowledge,
    tools=[SerperTools()],
    session_id="My serper chatbot March 2",
    add_history_to_context=True,
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent memory is: {chat_history}",
    markdown=True,
    debug_mode=False
)




In [14]:
# Normally we use chat bot like a query engine.  Here is a way to test some queries.
test_queries = [
    "Recommend a romantic comedy for date night.",
    "Who are the nominees for Best Picture at the 2026 Oscars?",
    "What are the latest Bollywood movies in 2025?",
    "Is Oppenheimer available on Netflix?",
    "Summarize the chat so far"
    ]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent.run(q, stream=False)
    print("\nBot:")
    print(response.content)
    print("-" * 80)


Query: Recommend a romantic comedy for date night.

Bot:
Great choice for a date night! To tailor a romantic comedy recommendation for you, could you share if you prefer a modern movie or maybe a classic romantic comedy? Or any other preference you have in mind for the tone or style?
--------------------------------------------------------------------------------

Query: Who are the nominees for Best Picture at the 2026 Oscars?

Bot:
It looks like you've asked again about the Best Picture nominees for the 2026 Oscars.

To confirm, some nominees include:
- Sinners
- One Battle After Another
- Frankenstein
- Einstein's Clocks
- The Stationmasters
- I Saw the TV Glow
- Rumi's Spin
- Canis Lupus
- Placid Edge

If you'd like more details about any of these films or want recommendations based on these nominees, just let me know!
--------------------------------------------------------------------------------

Query: What are the latest Bollywood movies in 2025?

Bot:
To recommend the latest

In [15]:
# This code will demonstrate a real flipped interaction

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes Samurai fiction"

Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction

Agent:
Great! For your date night, you enjoy romantic movies, and your spouse is into samurai fiction. That's a wonderful combination.

Would you prefer a movie that leans more into romance with a touch of samurai elements, or a samurai film with some romantic storyline? Or maybe a perfect blend of both? This will help me find the ideal movie for both of you.

Session Memory:
- I would like to watch movie associated with current wars.  Which is the latest war 
- I can help you with that! To clarify, by "current wars," are you referring to ongoing or very recent conflicts that have been depicted in movies? Also, are you looking for movies based on a specific recent war, or are you open to films based on any contemporary conflicts? And lastly, do you have a preferred genre or style for these war-related movies (e.g., drama, documentary, action)?

Could you please share